In [1]:
#%load_ext autoreload
#%autoreload 2

# Tutorial: Lorenz-96 Data Assimilation (ICESEE)

This notebook runs the Lorenz96 example using ICESEE's data assimilation driver.

In [ ]:
# Setup and preprocessing:

import os, sys
import platform
import numpy as np
from pathlib import Path

from IPython.display import HTML

# Current path
Path = os.path.realpath(".")
#print ('Path: %s\n' %Path)

# Parent path
Parentpath = os.path.dirname(Path)
#print ('Parentpath: %s\n' %Parentpath)

# Add to PYTHONPATH
sys.path.append(Parentpath)

# Tool path
Toolpath = os.path.dirname(Parentpath)
#print ('Toolpath: %s\n' %Toolpath)

import utils
#help(utils)

if utils.Configuration.VERBOSE == True:
    
    print ("Operating System Platform: " + platform.system(), platform.release())
    print ("\n")

    print ("os.environ['PATH']: ", os.environ['PATH'])
    print ("sys.path: ", sys.path)
    print ("\n")


AttributeError: module 'utils' has no attribute 'Configuration'

In [5]:
#https://api.jquery.com/ready/
HTML('''
<script>
    function scroll_to_top() {
        Jupyter.notebook.scroll_to_top();
    } 
    $( window ).on( "load", scroll_to_top() );
</script>
''')

In [2]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

<IPython.core.display.Javascript object>

In [3]:
print ('\nDirectory tree of\n%s:\n' %Toolpath)
results_tree = utils.Tree(Toolpath)
results_tree.print_tree()


NameError: name 'Toolpath' is not defined

In [ ]:
# --- Notebook directory ---
nb_dir = Path.cwd()  # assumes you're launching notebook from repo context
print("Notebook working directory:", nb_dir)

# --- Try to locate ICESEE submodule path (adjust if your layout differs) ---
# Typical: ICESEE-GHUB/external/ICESEE/src is where the Python package lives
candidate_paths = [
    nb_dir / "external" / "ICESEE" / "src",
    nb_dir.parent / "external" / "ICESEE" / "src",
    nb_dir.parent.parent / "external" / "ICESEE" / "src",
]

icesee_src = None
for p in candidate_paths:
    if p.exists():
        icesee_src = p
        break

if icesee_src is None:
    print("WARNING: Could not find external/ICESEE/src automatically.")
else:
    if str(icesee_src) not in sys.path:
        sys.path.insert(0, str(icesee_src))
    print("Using ICESEE src path:", icesee_src)

In [ ]:
from ICESEE.config._utility_imports import *
from ICESEE.config._utility_imports import params, kwargs, modeling_params, enkf_params, physical_params

from ICESEE.src.run_model_da.run_models_da import icesee_model_data_assimilation
from ICESEE.src.parallelization.parallel_mpi.icesee_mpi_parallel_manager import ParallelManager

In [ ]:
from ICESEE.applications.lorenz_model.examples.lorenz96._lorenz96_model import initialize_model

In [ ]:
rank, size, comm, _ = ParallelManager().icesee_mpi_init(params)
print(f"MPI rank/size: {rank}/{size}")

In [ ]:
# --- Ensemble Parameters ---
params.update({
    "nt": int(float(modeling_params["num_years"]) / float(modeling_params["dt"])),
    "nd": int(float(enkf_params["num_state_vars"]))
})

# --- model parameters ---
kwargs.update({
    "nt": params["nt"],
    "nd": params["nd"],
    "dt": float(modeling_params["dt"]),
    "seed": float(enkf_params["seed"]),
    "t": np.linspace(0, int(float(modeling_params["num_years"])), params["nt"] + 1),

    "u0True": np.array([1, 1, 1]),
    "u0b": np.array([2.0, 3.0, 4.0]),

    "sigma_96": float(physical_params["sigma_96"]),
    "beta_96": eval(physical_params["beta_96"]),  # keeping your original behavior
    "rho_96": float(physical_params["rho_96"]),
})

# --- Pass params into kwargs ---
kwargs.update({"params": params})

print("nt:", params["nt"], "nd:", params["nd"])
print("keys(kwargs):", list(kwargs.keys())[:15], "...")

### Call the Data Assimilation routine

In [6]:
icesee_model_data_assimilation(**kwargs)

NameError: name 'icesee_model_data_assimilation' is not defined